# Starter sample generation

Build SOLUSDT maker-trade samples from BTCUSDT/ETHUSDT/SOLUSDT market data files.

In [ ]:
from pathlib import Path
import importlib
import sys

ROOT = Path.cwd()
if not (ROOT / "utils").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import utils.starter_samples as starter_samples
starter_samples = importlib.reload(starter_samples)
build_samples = starter_samples.build_samples
toy_strategy_stats = starter_samples.toy_strategy_stats
monthly_profit_summary = starter_samples.monthly_profit_summary


# Folder created by ./copy_data.sh.
MARKET_DATA_PATH = ROOT / "data/market_data"
# First public date to generate samples for.
START_DATE = "20260201"
# Last public date to generate samples for.
END_DATE = "20260430"
# Events at or after sample time minus this latency are hidden from prediction.
LATENCY_US = 10_000
# Keep the first trade per maker direction in each window.
SELECT_WINDOW_US = 100_000
# If False, keep only trades strictly deeper than the prior best price.
BEST_PRICE_TRADES = True
# Maker rebate added to the 1s target, in basis points.
REBATE_BPS = 0.0
# Future horizon used for the profit target.
PROFIT_HORIZON_US = 1_000_000


Generate samples from cleaned market data, save them locally, and preview the first rows.

In [ ]:
# Number of dates processed in parallel. Lower this if RAM is tight.
WORKERS = 16
# Path to save the samples to
SAMPLES_OUTPUT_PATH = ROOT / "data/samples/starter_samples.parquet"

# One row per first SOLUSDT maker-side trade in each 100ms bucket.
samples = build_samples(
    market_data_root=MARKET_DATA_PATH,
    start_date=START_DATE,
    end_date=END_DATE,
    latency_us=LATENCY_US,
    select_window_us=SELECT_WINDOW_US,
    best_price_trades=BEST_PRICE_TRADES,
    rebate_bps=REBATE_BPS,
    profit_horizon_us=PROFIT_HORIZON_US,
    workers=WORKERS,
)
SAMPLES_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
samples.to_parquet(SAMPLES_OUTPUT_PATH, index=False, compression="zstd")
print(f"wrote {len(samples):,} rows to {SAMPLES_OUTPUT_PATH}")
monthly_profit_summary(samples)


Plot the 1-second profit distribution for all generated samples in basis points. The y-axis is logarithmic so the tails remain visible.

In [ ]:
import matplotlib.pyplot as plt

profit_bps = samples["profit_1s_with_rebate"] * 10_000.0
zero_count = int((profit_bps == 0.0).sum())
zero_fraction = zero_count / len(profit_bps)
print(f"zero-profit samples: {zero_count:,} / {len(profit_bps):,} ({zero_fraction:.2%})")

ax = profit_bps.hist(bins=300, figsize=(10, 4), log=True)
ax.set_title("1s maker profit distribution")
ax.set_xlabel("profit, bps")
ax.set_ylabel("sample count, log scale")
plt.show()

profit_bps.describe(percentiles=[0.01, 0.10, 0.50, 0.90, 0.99])


Run a toy momentum rule to show the expected evaluation shape by month. It selects maker bid trades when the recent mid-price move is above the threshold, and maker ask trades when the move is below the negative threshold.

In [ ]:
# Toy momentum rule: buy maker bid trades after an upward move, sell maker ask trades after a downward move.
toy_strategy_stats(samples, threshold_bps=0.0)